# MedRoute — Demonstração do Algoritmo Genético

Este notebook demonstra como usar o Algoritmo Genético para otimizar rotas de entrega de medicamentos.

## O que você aprenderá aqui:
1. Gerar um conjunto de cidades (pontos de entrega)
2. Executar o algoritmo genético
3. Visualizar a convergência
4. Comparar diferentes parâmetros

## Passo 1: Importar bibliotecas e módulos

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # Adiciona src/ ao path

import matplotlib.pyplot as plt
import numpy as np
from ga.algorithm import run_ga_visual
from ga.fitness import calculate_distance, calculate_fitness
from ga.cities import generate_cities, load_att48

print("✅ Módulos importados com sucesso!")

## Passo 2: Gerar ou carregar cidades

In [ ]:
# Opção 1: Gerar cidades aleatórias
num_cities = 30
cities = generate_cities(num_cities)

print(f"✅ {num_cities} cidades geradas aleatoriamente")
print(f"   Primeiras 5: {cities[:5]}")

# Opção 2: Carregar dados reais (se disponível)
# cities = load_att48()
# print(f"✅ Dataset ATT48 carregado com {len(cities)} cidades")

## Passo 3: Visualizar as cidades no mapa

In [ ]:
# Plota os pontos de entrega
plt.figure(figsize=(10, 8))
x_coords = [city[0] for city in cities]
y_coords = [city[1] for city in cities]

plt.scatter(x_coords, y_coords, c='red', s=100, alpha=0.6, edgecolors='darkred')
plt.title(f'Mapa de Entrega ({num_cities} pontos)', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)

# Adiciona rótulos nas cidades
for i, (x, y) in enumerate(cities):
    plt.annotate(str(i), (x, y), fontsize=8, ha='right')

plt.tight_layout()
plt.show()

print(f"✅ Mapa gerado com {len(cities)} pontos de entrega")

## Passo 4: Executar o Algoritmo Genético

In [ ]:
# Configurar parâmetros
params = {
    "population_size": 200,
    "mutation_probability": 0.2,
    "tournament_k": 3,
    "max_generations": 150,
    "selection_type": "tournament",  # "tournament", "top10", "roulette"
    "distance_function": calculate_distance
}

print("🚀 Executando Algoritmo Genético...")
print(f"   Configuração: {params}\n")

# Executa o GA
history, best_routes, populations = run_ga_visual(cities, **params)

print("\n✅ GA concluído!")
print(f"   Distância inicial: {history[0]:.2f}")
print(f"   Distância final:   {history[-1]:.2f}")
print(f"   Melhoria:          {((history[0] - history[-1]) / history[0] * 100):.2f}%")

## Passo 5: Analisar Convergência

In [ ]:
# Gráfico de convergência
plt.figure(figsize=(12, 5))

# Subplot 1: Evolução da melhor distância
plt.subplot(1, 2, 1)
plt.plot(history, linewidth=2, color='#0ea5e9')
plt.fill_between(range(len(history)), history, alpha=0.3, color='#0ea5e9')
plt.xlabel('Geração')
plt.ylabel('Distância Total (km)')
plt.title('Convergência do Algoritmo Genético')
plt.grid(True, alpha=0.3)

# Subplot 2: Taxa de melhoria por geração
plt.subplot(1, 2, 2)
improvement = [(history[i-1] - history[i]) for i in range(1, len(history))]
plt.bar(range(len(improvement)), improvement, color='#00c896', alpha=0.7)
plt.xlabel('Geração')
plt.ylabel('Melhoria (km)')
plt.title('Taxa de Melhoria por Geração')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Passo 6: Visualizar a Melhor Rota

In [ ]:
# Obtém a melhor rota (última geração)
best_route = best_routes[-1]
best_distance = history[-1]

# Plota a rota
plt.figure(figsize=(10, 8))

# Plotar cidades
x_coords = [city[0] for city in cities]
y_coords = [city[1] for city in cities]
plt.scatter(x_coords, y_coords, c='red', s=100, alpha=0.6, edgecolors='darkred', zorder=3)

# Plotar rota (linha conectando as cidades na ordem)
route_x = [cities[i][0] for i in best_route] + [cities[best_route[0]][0]]
route_y = [cities[i][1] for i in best_route] + [cities[best_route[0]][1]]
plt.plot(route_x, route_y, 'b-', linewidth=1.5, alpha=0.7, zorder=1)

# Plotar seta de início
plt.arrow(route_x[0], route_y[0], route_x[1]-route_x[0], route_y[1]-route_y[0],
         head_width=2, head_length=1.5, fc='green', ec='green', zorder=2)

plt.title(f'Melhor Rota Encontrada\nDistância: {best_distance:.2f} km', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ Melhor rota: {best_route}")
print(f"   Distância total: {best_distance:.2f} km")

## Passo 7: Comparar Diferentes Configurações

In [ ]:
# Testar diferentes configurações de seleção
selection_types = ["tournament", "top10", "roulette"]
results_by_selection = {}

for selection in selection_types:
    print(f"\n🔄 Testando seleção: {selection}...")
    
    history, _, _ = run_ga_visual(
        cities,
        population_size=200,
        mutation_probability=0.2,
        tournament_k=3,
        max_generations=100,
        selection_type=selection,
        distance_function=calculate_distance
    )
    
    results_by_selection[selection] = history
    print(f"   ✅ Melhor distância: {history[-1]:.2f}")

# Plota comparação
plt.figure(figsize=(12, 6))

for selection, history in results_by_selection.items():
    plt.plot(history, label=selection, linewidth=2)

plt.xlabel('Geração')
plt.ylabel('Distância Total (km)')
plt.title('Comparação de Métodos de Seleção')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Passo 8: Extrair Métricas

In [ ]:
# Calcula estatísticas da corrida do GA
print("📊 ESTATÍSTICAS DA EXECUÇÃO\n")
print(f"Número de gerações:    {len(history)}")
print(f"Melhor distância:      {min(history):.2f} km")
print(f"Pior distância:        {max(history):.2f} km")
print(f"Distância média:       {np.mean(history):.2f} km")
print(f"Desvio padrão:         {np.std(history):.2f} km")
print(f"\nTaxa de convergência:  {((history[0] - history[-1]) / history[0] * 100):.2f}%")
print(f"Número de cidades:     {len(cities)}")
print(f"Tamanho da população:  {params['population_size']}")
print(f"Probabilidade mutação: {params['mutation_probability']}")